# Objects, definitions, and repositories

This standalone lesson requires the matching installed DRYML version and only the base installation. It runs offline, creates Stores only in temporary directories, and leaves no files behind.

A `Definition` is an unresolved construction recipe. Concretizing it produces a hashable `ConcreteDefinition`, which is the stable identity attached to a live `Object`. Values produced after construction are mutable state, not identity.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from dryml.core2 import ConcreteDefinition, Definition, Object, Repo, SKIP_ARGS
from dryml.core2.object import Pickleable
from dryml.core2.query import QueryDomainError
from dryml.core2.store import DirStore


class Counter(Object):
    """Mutable counter whose constructor value determines identity."""

    def __init__(self, start=0):
        """Initialize the counter value from start."""
        super().__init__()
        self.value = start

    def increment(self):
        """Increase the mutable value by one and return None."""
        self.value += 1


class Envelope(Object):
    """Object that gives a child a queryable owner and label."""

    def __init__(self, child, label):
        """Store the child object and its descriptive label."""
        super().__init__()
        self.child = child
        self.label = label


## Constructor identity versus mutable state

Normal construction still records the same concrete identity as an explicit recipe. Mutating `value` does not change that constructor-derived identity, and rebuilding in a fresh Repo starts from the constructor value rather than copying the mutation.

In [ ]:
identity_repo = Repo()
fresh_repo = Repo()
try:
    recipe = Definition(Counter, start=2)
    expected_identity = recipe.concretize(repo=identity_repo)
    counter = Counter(start=2, repo=identity_repo)

    assert isinstance(recipe, Definition)
    assert isinstance(expected_identity, ConcreteDefinition)
    assert counter.definition == expected_identity

    counter.increment()
    assert counter.value == 3
    assert counter.definition == expected_identity

    rebuilt = expected_identity.build(repo=fresh_repo, instance='new', cache='none', restore_state=False)
    assert rebuilt.definition == expected_identity
    assert rebuilt.value == 2
finally:
    identity_repo.close(flush=False)
    fresh_repo.close(flush=False)


## Save, close, reopen, and load

Portable persistence targets must be importable from the installed package. The public `Pickleable` object below has no tutorial support module: its empty constructor determines identity, while attributes added afterward are saved state. An alias is another reference to the same definition, not another identity. Temporary Stores are same-version demonstrations rather than interchange archives.

In [ ]:
with TemporaryDirectory() as temporary_root:
    store_path = Path(temporary_root) / 'portable-store'
    repo = Repo(stores=DirStore(store_path, query_index='memory'))
    stateful = Pickleable(repo=repo)
    stateful.message = 'saved mutable state'
    exact_definition = stateful.definition
    assert exact_definition == Definition(Pickleable).concretize(repo=repo)

    repo.save_object(stateful, alias='lesson-state')
    repo.close()

    reopened = Repo(stores=DirStore(store_path, query_index='memory'))
    try:
        exact_loaded = reopened.load(exact_definition, instance='new', cache='none')
        alias_loaded = reopened.load_alias('lesson-state', instance='new', cache='none')
        assert exact_loaded.definition == exact_definition
        assert alias_loaded.definition == exact_definition
        assert exact_loaded.message == 'saved mutable state'
        assert alias_loaded.message == 'saved mutable state'
    finally:
        reopened.close(flush=False)


## Query explicit domains

A saved root and a definition nested inside its constructor graph occupy different query domains. `.stored()` returns independently stored roots. `.nested().definitions()` returns matching nested definitions, which cannot be materialized directly. `.nested().owners()` projects those occurrences back to stored owner definitions. The local `Counter` and `Envelope` are used only for this in-process graph lesson, not as the portable persistence target above.

In [ ]:
with TemporaryDirectory() as temporary_root:
    query_store = DirStore(Path(temporary_root) / 'query-store', query_index='memory')
    query_repo = Repo(stores=query_store)
    try:
        child = Counter(start=4, repo=query_repo)
        owner = Envelope(child, label='owner', repo=query_repo)
        query_repo.save_object(owner)

        stored_roots = query_repo.query(Definition(Envelope, SKIP_ARGS)).stored().defs()
        nested_definitions = (
            query_repo.query(Definition(Counter, SKIP_ARGS))
            .nested()
            .definitions()
            .defs()
        )
        owner_definitions = (
            query_repo.query(Definition(Counter, SKIP_ARGS))
            .nested()
            .owners()
            .defs()
        )

        assert stored_roots.domain == 'stored', 'stored query domain'
        assert nested_definitions.domain == 'nested-definitions', 'nested definition domain'
        assert owner_definitions.domain == 'owners', 'owner query domain'
        assert list(stored_roots) == [owner.definition], 'stored roots'
        assert list(nested_definitions) == [child.definition], 'nested definitions'
        assert list(owner_definitions) == [owner.definition], 'nested owners'
        assert query_store.has(owner.definition), 'owner is stored'
        assert not query_store.has(child.definition), 'child is nested but not stored'
    finally:
        query_repo.close()


A terminal is intentionally invalid until a domain is selected. Handle `QueryDomainError` when demonstrating that boundary so an expected teaching error does not become notebook traceback output.

In [ ]:
error_repo = Repo()
try:
    try:
        error_repo.query(None).defs()
    except QueryDomainError as error:
        assert 'Select a query domain' in str(error)
    else:
        raise AssertionError('a query terminal without a domain must fail')
finally:
    error_repo.close(flush=False)
